In [0]:
df_patient = spark.table("workspace.new.data_2")
df_patient.display()

patient_id,name,age,gender,department,city,admission_type
P001,Arjun,45,Male,Cardiology,Kochi,Emergency
P002,Meera,32,Female,Neurology,Kozhikode,Planned
P003,Rahul,28,Male,Orthopedics,Trivandrum,Emergency
P004,Anjali,52,Female,Cardiology,Kochi,Planned
P005,Vikram,36,Male,General Medicine,Thrissur,Emergency
P006,Divya,41,Female,Neurology,Kochi,Planned
P007,Sneha,25,Female,Orthopedics,Kozhikode,Emergency
P008,Kiran,60,Male,Cardiology,Trivandrum,Emergency
P009,Asha,38,Female,General Medicine,Thrissur,Planned
P010,Manoj,47,Male,Cardiology,Kochi,Emergency


# Multi-Level Thinking
 - Find: Patient count by department AND city
 - Output should show:
     department
     city
     patient_count

In [0]:
from pyspark.sql.functions import count
df_patient.groupBy("department", "city").agg(count("*").alias("patient_count")).show()

+----------------+----------+-------------+
|      department|      city|patient_count|
+----------------+----------+-------------+
|      Cardiology|     Kochi|            3|
|       Neurology| Kozhikode|            1|
|     Orthopedics|Trivandrum|            1|
|General Medicine|  Thrissur|            2|
|       Neurology|     Kochi|            2|
|     Orthopedics| Kozhikode|            1|
|      Cardiology|Trivandrum|            1|
|     Orthopedics|  Thrissur|            1|
+----------------+----------+-------------+



# Smart Aggregation
 - Find Average age of patients for each admission_type
 - Output:
    admission_type
    avg_age

In [0]:
from pyspark.sql.functions import avg
df_patient.groupBy("admission_type").agg(avg("age").alias("avg_age")).show()

+--------------+-----------------+
|admission_type|          avg_age|
+--------------+-----------------+
|     Emergency|38.57142857142857|
|       Planned|             43.6|
+--------------+-----------------+



# Filter After Aggregation (Important)
- Find Departments where average age > 40
- Output:
    department
    avg_age

In [0]:
df_patient.groupBy("department").agg(avg("age").alias("avg_age")).filter("avg_age > 40").show()

+----------+-------+
|department|avg_age|
+----------+-------+
|Cardiology|   51.0|
+----------+-------+



# Top Performer (Real Analytics)
- Find City with highest number of patients
- Output:
    city
    count

In [0]:
from pyspark.sql.functions import desc
df_patient.groupBy("city").agg(count("city").alias("count")).orderBy(desc("count")).show(1)

+-----+-----+
| city|count|
+-----+-----+
|Kochi|    5|
+-----+-----+
only showing top 1 row


# Multiple Aggregations
- Find for each department: Total patients, Average age
- Output: department, total_patients, avg_age

In [0]:
df_patient.groupBy("department").agg(count("*").alias("total_patients"),avg("age").alias("avg_age")).show()

+----------------+--------------+-------+
|      department|total_patients|avg_age|
+----------------+--------------+-------+
|      Cardiology|             4|   51.0|
|       Neurology|             3|   34.0|
|     Orthopedics|             3|   36.0|
|General Medicine|             2|   37.0|
+----------------+--------------+-------+



# Business Insight (Slightly Tricky)
- Find Number of Emergency vs Planned patients in each city
- Output:
    city
    admission_type
    count
 

In [0]:
df_patient.groupBy("city","admission_type").agg(count("*").alias("count")).show()

+----------+--------------+-----+
|      city|admission_type|count|
+----------+--------------+-----+
|     Kochi|     Emergency|    3|
| Kozhikode|       Planned|    1|
|Trivandrum|     Emergency|    2|
|     Kochi|       Planned|    2|
|  Thrissur|     Emergency|    1|
| Kozhikode|     Emergency|    1|
|  Thrissur|       Planned|    2|
+----------+--------------+-----+



# Conditional Thinking 
- Find Number of patients above age 40 in each department
- Output: department, count_above_40

In [0]:
from pyspark.sql.functions import count
df_patient.filter(df_patient.age > 40).groupBy("department").agg(count("*").alias("count_above_40")).show()

+-----------+--------------+
| department|count_above_40|
+-----------+--------------+
| Cardiology|             4|
|  Neurology|             1|
|Orthopedics|             1|
+-----------+--------------+



In [0]:
from pyspark.sql.functions import when,count
df_patient.groupBy("department").agg(count(when(df_patient.age > 40,1)).alias("count_above_40")).show()

+----------------+--------------+
|      department|count_above_40|
+----------------+--------------+
|      Cardiology|             4|
|       Neurology|             1|
|     Orthopedics|             1|
|General Medicine|             0|
+----------------+--------------+



# Double Condition Aggregation
- Find For each department Count of Emergency patients only
- Output: department, emergency_count

In [0]:
df_patient.filter(df_patient.admission_type == "Emergency").groupBy("department").agg(count("*").alias("emergency_count")).show()

+----------------+---------------+
|      department|emergency_count|
+----------------+---------------+
|      Cardiology|              3|
|     Orthopedics|              2|
|General Medicine|              1|
|       Neurology|              1|
+----------------+---------------+



In [0]:
from pyspark.sql.functions import when,count
df_patient.groupBy("department").agg(count(when(df_patient.admission_type == "Emergency",1)).alias("emergency_count")).show()

+----------------+---------------+
|      department|emergency_count|
+----------------+---------------+
|      Cardiology|              3|
|       Neurology|              1|
|     Orthopedics|              2|
|General Medicine|              1|
+----------------+---------------+



# Ranking Logic 
- Find Top 2 departments with highest patient count
- Output:
    department
    count

In [0]:
from pyspark.sql.functions import desc
df_patient.groupBy("department").agg(count("*").alias("department_count")).orderBy(desc("department_count")).show(2)

+----------+----------------+
|department|department_count|
+----------+----------------+
|Cardiology|               4|
| Neurology|               3|
+----------+----------------+
only showing top 2 rows


# Insight Question 
- “Which department should get more resources based on patient count?”

Run analysis
Give reasoning

In [0]:
df_patient.groupBy("department").agg(count("*").alias("department_count")).orderBy(desc("department_count")).show(1)
print("The department shown above should get more resources as it has the highest patient count, indicating greater demand for services.")

+----------+----------------+
|department|department_count|
+----------+----------------+
|Cardiology|               4|
+----------+----------------+
only showing top 1 row
The department shown above should get more resources as it has the highest patient count, indicating greater demand for services.


In [0]:
df_patient.groupBy("department","admission_type").agg(count("*").alias("department_count")).orderBy(desc("department_count")).show()

+----------------+--------------+----------------+
|      department|admission_type|department_count|
+----------------+--------------+----------------+
|      Cardiology|     Emergency|               3|
|     Orthopedics|     Emergency|               2|
|       Neurology|       Planned|               2|
|General Medicine|       Planned|               1|
|      Cardiology|       Planned|               1|
|General Medicine|     Emergency|               1|
|       Neurology|     Emergency|               1|
|     Orthopedics|       Planned|               1|
+----------------+--------------+----------------+



In [0]:
df_patient.filter((df_patient.age > 30) & (df_patient.city == "Kochi")).show()

+----------+------+---+------+----------+-----+--------------+
|patient_id|  name|age|gender|department| city|admission_type|
+----------+------+---+------+----------+-----+--------------+
|      P001| Arjun| 45|  Male|Cardiology|Kochi|     Emergency|
|      P004|Anjali| 52|Female|Cardiology|Kochi|       Planned|
|      P006| Divya| 41|Female| Neurology|Kochi|       Planned|
|      P010| Manoj| 47|  Male|Cardiology|Kochi|     Emergency|
+----------+------+---+------+----------+-----+--------------+



In [0]:
from pyspark.sql.functions import lit
df_patient.withColumn("country", lit("India")).show()

+----------+------+---+------+----------------+----------+--------------+-------+
|patient_id|  name|age|gender|      department|      city|admission_type|country|
+----------+------+---+------+----------------+----------+--------------+-------+
|      P001| Arjun| 45|  Male|      Cardiology|     Kochi|     Emergency|  India|
|      P002| Meera| 32|Female|       Neurology| Kozhikode|       Planned|  India|
|      P003| Rahul| 28|  Male|     Orthopedics|Trivandrum|     Emergency|  India|
|      P004|Anjali| 52|Female|      Cardiology|     Kochi|       Planned|  India|
|      P005|Vikram| 36|  Male|General Medicine|  Thrissur|     Emergency|  India|
|      P006| Divya| 41|Female|       Neurology|     Kochi|       Planned|  India|
|      P007| Sneha| 25|Female|     Orthopedics| Kozhikode|     Emergency|  India|
|      P008| Kiran| 60|  Male|      Cardiology|Trivandrum|     Emergency|  India|
|      P009|  Asha| 38|Female|General Medicine|  Thrissur|       Planned|  India|
|      P010| Man

In [0]:
df_patient.groupBy("department").count().orderBy("count")

DataFrame[department: string, count: bigint]